In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import torch
import re

from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer  # For Tokenization for BERT

In [2]:
# Check for GPU
print(torch.cuda.is_available())

True


In [3]:
# store the gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device :  {device}")  

Using Device :  cuda


In [4]:
# Load the data

df = pd.read_csv("IMDB.csv")
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [5]:
df.shape

(50000, 2)

In [6]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
# Label Encoding
df['label'] = df['sentiment'].map({'positive':1, 'negative':0})
print(df.head())

                                              review sentiment  label
0  One of the other reviewers has mentioned that ...  positive      1
1  A wonderful little production. <br /><br />The...  positive      1
2  I thought this was a wonderful way to spend ti...  positive      1
3  Basically there's a family where a little boy ...  negative      0
4  Petter Mattei's "Love in the Time of Money" is...  positive      1


In [9]:
# Messy Inputs
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [10]:
# data proprocessing : Cleaning the text

def clean_text(text):
    
    # Remove html tags
    text = re.sub(r'<.*?>', "", text)   # .*? : anything - characters, numbers, symbols
    
    # Remove numbers
    text = re.sub(r'\d+', "", text)
    
    # Lower case
    text = text.lower()
    
    return text

In [11]:
# Clean dataset text
df['cleaned_review'] = df['review'].apply(clean_text)
print(df['cleaned_review'][0])

one of the other reviewers has mentioned that after watching just  oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.i would say the main appeal of the show is due to the fact that it goes where other shows woul

In [12]:
print("BEFORE:\n", df['review'][0])
print("\nAFTER:\n", df['cleaned_review'][0])


BEFORE:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due t

In [13]:
# Split the data

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['cleaned_review'].values,
    df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label'].values
)

In [14]:
# Split further into validation and test data

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

In [15]:
print(f"Train size: {len(train_texts)}")
print(f"Validation size: {len(val_texts)}")
print(f"Test size: {len(test_texts)}")

Train size: 40000
Validation size: 5000
Test size: 5000


In [16]:
print(f"Train label distribution: {np.unique(train_labels, return_counts=True)}")
print(f"Val label distribution: {np.unique(val_labels, return_counts=True)}")
print(f"Test label distribution: {np.unique(test_labels, return_counts=True)}")

Train label distribution: (array([0, 1]), array([20000, 20000]))
Val label distribution: (array([0, 1]), array([2500, 2500]))
Test label distribution: (array([0, 1]), array([2500, 2500]))


In [ ]:
# Tokenization
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')  # Bert's Tokenizer

d:\Prem\Codes\Innomatics Internship\innomatics-genai-internship-2026\IN226105802_NLP\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\premv\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [20]:
sample_text = "I love this man prem !!"
tokens = tokenizer(sample_text)
print(tokens)

{'input_ids': [101, 1045, 2293, 2023, 2158, 26563, 999, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [21]:
print(tokenizer.convert_ids_to_tokens(tokens['input_ids']))

['[CLS]', 'i', 'love', 'this', 'man', 'prem', '!', '!', '[SEP]']
